In [6]:
import urllib.request
import os

# Создаем папку для JAR-файлов
os.makedirs("/home/jovyan/jars", exist_ok=True)

# Скачиваем hadoop-aws
urllib.request.urlretrieve(
    "https://repo1.maven.org/maven2/org/apache/hadoop/hadoop-aws/3.3.4/hadoop-aws-3.3.4.jar",
    "/home/jovyan/jars/hadoop-aws-3.3.4.jar"
)

# Скачиваем aws-java-sdk
urllib.request.urlretrieve(
    "https://repo1.maven.org/maven2/com/amazonaws/aws-java-sdk-bundle/1.12.262/aws-java-sdk-bundle-1.12.262.jar",
    "/home/jovyan/jars/aws-java-sdk-bundle-1.12.262.jar"
)

print("✅ JAR-файлы загружены в /home/jovyan/jars/")

✅ JAR-файлы загружены в /home/jovyan/jars/


In [1]:
from pyspark.sql import SparkSession
import os

# Очистим старую SparkSession если она была
try:
    spark.stop()
except:
    pass

# Создаем новую SparkSession
spark = SparkSession.builder \
    .appName("MinIO Lakehouse") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .getOrCreate()

print("✅ SparkSession создана!")
print(f"Spark version: {spark.version}")

# Проверяем, что JAR-файлы загружены
try:
    spark._jvm.org.apache.hadoop.fs.s3a.S3AFileSystem
    print("✅ S3AFileSystem доступен!")
except Exception as e:
    print(f"❌ S3AFileSystem НЕ доступен: {e}")

✅ SparkSession создана!
Spark version: 3.5.0
✅ S3AFileSystem доступен!


In [2]:
# Импортируем boto3 для работы с MinIO
!pip install boto3 -q

import boto3
from botocore.client import Config

# Подключаемся к MinIO
s3_client = boto3.client(
    's3',
    endpoint_url='http://minio:9000',
    aws_access_key_id='admin',
    aws_secret_access_key='password',
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

# Список всех бакетов
buckets = s3_client.list_buckets()
print("📦 Доступные бакеты в MinIO:")
for bucket in buckets['Buckets']:
    print(f"  - {bucket['Name']}")

📦 Доступные бакеты в MinIO:
  - lakehouse
  - warehouse


In [3]:
# Создаем тестовый DataFrame
test_data = [
    (1, "Взрыв насоса", "2024-01-15", "Нефтеперерабатывающий завод"),
    (2, "Утечка газа", "2024-02-20", "Газораспределительная станция"),
    (3, "Обрушение конструкции", "2024-03-10", "Строительная площадка"),
]

columns = ["incident_id", "description", "date", "location"]

df_incidents = spark.createDataFrame(test_data, columns)

print("📊 Тестовые данные созданы:")
df_incidents.show()

📊 Тестовые данные созданы:
+-----------+--------------------+----------+--------------------+
|incident_id|         description|      date|            location|
+-----------+--------------------+----------+--------------------+
|          1|        Взрыв насоса|2024-01-15|Нефтеперерабатыва...|
|          2|         Утечка газа|2024-02-20|Газораспределител...|
|          3|Обрушение констру...|2024-03-10|Строительная площ...|
+-----------+--------------------+----------+--------------------+



In [4]:
# Путь в MinIO (S3)
lakehouse_path = "s3a://lakehouse/incidents/"

# Сохраняем в формате Parquet
df_incidents.write.mode("overwrite").parquet(lakehouse_path)

print(f"✅ Данные сохранены в: {lakehouse_path}")

✅ Данные сохранены в: s3a://lakehouse/incidents/


In [9]:
# Читаем данные из MinIO
df_read = spark_nessie.read.parquet(lakehouse_path)

print("📖 Данные, прочитанные из MinIO:")
df_read.show()

# Проверяем количество записей
print(f"Всего инцидентов: {df_read.count()}")

📖 Данные, прочитанные из MinIO:
+-----------+--------------------+----------+--------------------+
|incident_id|         description|      date|            location|
+-----------+--------------------+----------+--------------------+
|          3|Обрушение констру...|2024-03-10|Строительная площ...|
|          2|         Утечка газа|2024-02-20|Газораспределител...|
|          1|        Взрыв насоса|2024-01-15|Нефтеперерабатыва...|
+-----------+--------------------+----------+--------------------+

Всего инцидентов: 3


In [6]:
# Просмотр объектов в бакете lakehouse
response = s3_client.list_objects_v2(Bucket='lakehouse')

print("📄 Объекты в бакете 'lakehouse':")
if 'Contents' in response:
    for obj in response['Contents']:
        print(f"  - {obj['Key']} (размер: {obj['Size']} байт)")
else:
    print("  Бакет пуст или ещё не содержит объектов")

📄 Объекты в бакете 'lakehouse':
  - incidents/_SUCCESS (размер: 0 байт)
  - incidents/part-00000-063b4af8-891b-479f-afea-eb895b277bfe-c000.snappy.parquet (размер: 583 байт)
  - incidents/part-00005-063b4af8-891b-479f-afea-eb895b277bfe-c000.snappy.parquet (размер: 1758 байт)
  - incidents/part-00010-063b4af8-891b-479f-afea-eb895b277bfe-c000.snappy.parquet (размер: 1777 байт)
  - incidents/part-00015-063b4af8-891b-479f-afea-eb895b277bfe-c000.snappy.parquet (размер: 1805 байт)


In [7]:
# Пример: группировка по локациям
from pyspark.sql.functions import col, count

location_counts = df_read.groupBy("location").count()
print("📊 Количество инцидентов по локациям:")
location_counts.show()

# Регистрируем как временное представление для SQL-запросов
df_read.createOrReplaceTempView("incidents")

# SQL-запрос
sql_result = spark.sql("""
    SELECT location, COUNT(*) as incident_count 
    FROM incidents 
    GROUP BY location 
    ORDER BY incident_count DESC
""")

print("📊 Результат SQL-запроса:")
sql_result.show()

📊 Количество инцидентов по локациям:
+--------------------+-----+
|            location|count|
+--------------------+-----+
|Строительная площ...|    1|
|Газораспределител...|    1|
|Нефтеперерабатыва...|    1|
+--------------------+-----+

📊 Результат SQL-запроса:
+--------------------+--------------+
|            location|incident_count|
+--------------------+--------------+
|Строительная площ...|             1|
|Газораспределител...|             1|
|Нефтеперерабатыва...|             1|
+--------------------+--------------+



In [8]:
# Настройка Spark для работы с Nessie
spark_nessie = SparkSession.builder \
    .appName("Nessie Lakehouse") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.sql.catalog.nessie", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.nessie.catalog-impl", "org.apache.iceberg.nessie.NessieCatalog") \
    .config("spark.sql.catalog.nessie.uri", "http://nessie:19120/api/v1") \
    .config("spark.sql.catalog.nessie.ref", "main") \
    .config("spark.sql.catalog.nessie.warehouse", "s3a://warehouse/") \
    .config("spark.sql.catalog.nessie.s3.endpoint", "http://minio:9000") \
    .config("spark.sql.catalog.nessie.s3.access-key-id", "admin") \
    .config("spark.sql.catalog.nessie.s3.secret-access-key", "password") \
    .getOrCreate()

print("✅ Spark с Nessie настроен!")

✅ Spark с Nessie настроен!
